In [3]:
import os
import sys
import yaml
import torch
from pathlib import Path
from copy import deepcopy
%pip install polars

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
# --- STEP 1: DEFINE EXACT PATHS ---
# Based on your screenshots:
parent_dir = Path("/home/FASTLAB2/Tamoor")
code_container = parent_dir / "Taimoor folder" / "ultralytics" 
# This is the folder that CONTAINS the 'ultralytics' package folder
package_parent = str(code_container) 
print(f"📁 Root: {parent_dir}")
print(f"💻 Adding to Path: {package_parent}")

📁 Root: /home/FASTLAB2/Tamoor
💻 Adding to Path: /home/FASTLAB2/Tamoor/Taimoor folder/ultralytics


In [5]:
# --- STEP 2: FORCE USE CUSTOM CODE ---

if package_parent not in sys.path:
    sys.path.insert(0, package_parent)
# Nuclear reset of any old YOLO versions in memory
for mod in list(sys.modules.keys()):
    if mod.startswith('ultralytics'):
        del sys.modules[mod]
    

In [6]:
# --- STEP 3: REGISTER CUSTOM MODULE (Fixes KeyError) ---
try:
    # We manually import your custom PC_C2f from the deep folder
    
    import ultralytics.nn.modules.block as block
    import ultralytics.nn.tasks as tasks
    
    # Inject it directly into the model parser
    tasks.PC_C2f = block.PC_C2f
    globals()['PC_C2f'] = block.PC_C2f
    
    from ultralytics import YOLO
    import ultralytics
    print(f"🚀 SUCCESS: YOLO loaded from: {ultralytics.__file__}")
except Exception as e:
    print(f"❌ Error linking custom modules: {e}")



🚀 SUCCESS: YOLO loaded from: /home/FASTLAB2/Tamoor/Taimoor folder/ultralytics/ultralytics/__init__.py


In [7]:
# --- STEP 4: SETUP DATA.YAML ---

dataset_root = parent_dir / "yolo_dataset"
new_yaml = parent_dir / "linux_data_fixed.yaml"
original_yaml = dataset_root / "data.yaml"
if original_yaml.exists():
    with open(original_yaml, 'r') as f:
        config = yaml.safe_load(f)
    config['path'] = str(dataset_root)
    with open(new_yaml, 'w') as f:
        yaml.dump(config, f)
    print("✅ data.yaml paths are fixed.")


✅ data.yaml paths are fixed.


In [8]:
# --- STEP 5: START TRAINING ---
# Find the exact model config path from your image
model_config = Path(package_parent) / "ultralytics" / "cfg" / "models" / "v8" / "yolov8.yaml"
if model_config.exists():
    print(f"--- Loading Model: {model_config} ---")
    model = YOLO(str(model_config))
    
    device = 0 if torch.cuda.is_available() else 'cpu'
    print(f"🖥️ Device: {device} ({'GPU Active' if device == 0 else 'CPU Only'})")
    
    #New addition
    
    # project_path = str(parent_dir/"runs")
    # experiment_name = "CAVC_Final_Training"
    results = model.train(
        data=str(new_yaml),
        epochs=20,
        imgsz=640,
        batch=16,
        device=device,
        optimizer='AdamW',
        cos_lr=True,
        project=str(parent_dir / "runs"),
        name="CAVC_Final_Training",
        
        #Checkpoint & validation
        
        save=True,
        save_period = 5,
        val = True,
        plots = True,
        exist_ok = True,
        
        #Augmentation & Performance
        
        workers = 8,
        cache = False
        
    )
else:
    print(f"❌ ERROR: yolov8.yaml not found at {model_config}")

--- Loading Model: /home/FASTLAB2/Tamoor/Taimoor folder/ultralytics/ultralytics/cfg/models/v8/yolov8.yaml ---
WARNING ⚠️ no model scale passed. Assuming scale='n'.
🖥️ Device: 0 (GPU Active)
New https://pypi.org/project/ultralytics/8.4.7 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.248 🚀 Python-3.8.10 torch-2.4.0+cu121 CUDA:0 (NVIDIA A40, 45416MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/home/FASTLAB2/Tamoor/linux_data_fixed.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj

       5/20      2.59G  3.039e-05      1.406      1.587          8        640: 100% ━━━━━━━━━━━━ 4008/4008 11.9it/s 5:38<0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 85/85 9.1it/s 9.3s0.1s
                   all       2693      10777      0.716      0.533      0.614      0.362

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       6/20      2.59G  2.841e-05       1.35      1.556         46        640: 100% ━━━━━━━━━━━━ 4008/4008 11.9it/s 5:38<0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 85/85 9.4it/s 9.0s0.1s
                   all       2693      10777      0.722      0.548      0.631      0.378

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       7/20      2.59G  2.698e-05      1.313      1.532         37        640: 100% ━━━━━━━━━━━━ 4008/4008 11.9it/s 5:37<0.1s
                 Class  